# Protein Context and Druggability: IDG/Pharos


## What do IDG and Pharos add?

GTEx and HuBMAP added expression context to genes from the heart-failure paper.
We now ask what is known about the proteins those genes encode.

IDG stands for **Illuminating the Druggable Genome**. This NIH Common Fund
program develops knowledge and research tools for understudied proteins in
druggable protein families. Pharos is an IDG resource that combines information
about human proteins and drug targets.

Pharos uses four target development levels, called TDLs:

- **Tclin:** linked to the action of an approved drug
- **Tchem:** has qualifying small-molecule activity data
- **Tbio:** has biological information but does not meet Tclin or Tchem rules
- **Tdark:** has limited information and belongs to a selected target family

TDL describes a protein target. It does not rank the paper's variants or change
their study classifications.

## How the Pharos API Works

Pharos uses GraphQL. The request names the exact target fields to return. The
helper repeats that query for each of the paper's 25 gene symbols and puts the
responses in one table.

First, load the published variants and the shared API helper.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd() if Path("api_helpers.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from api_helpers import fetch_pharos_context

DATA_DIR = REPO_ROOT / "data"
variants = pd.read_csv(DATA_DIR / "variants.csv")


Next, query Pharos for the 25 unique genes. The commented line loads the saved
response if the live service is unavailable.


In [ ]:
gene_symbols = sorted(variants["gene_symbol"].unique())
pharos = fetch_pharos_context(gene_symbols)

# Backup: use the frozen 2026-08-11 response instead of the live API.
# pharos = pd.read_csv(DATA_DIR / "pharos_target_context.csv")

pharos.head()


### Live data and backup
The default code queries Pharos. Target information can change. If the request
fails, comment out the live line and uncomment the saved-data line.

## Review the 25 Protein Targets

Return to the paper's unique genes and compare their protein information.

First, count the target development levels in the current Pharos response.


In [ ]:
tdl_summary = (
    pharos["tdl"]
    .value_counts()
    .rename_axis("tdl")
    .reset_index(name="genes")
)
tdl_summary


Next, keep the target fields used for interpretation and order the proteins by
TDL, drug count, and gene symbol.


In [ ]:
tdl_order = pd.CategoricalDtype(
    categories=["Tclin", "Tchem", "Tbio", "Tdark"],
    ordered=True,
)
target_summary = (
    pharos.loc[
        :,
        [
            "gene_symbol",
            "target_name",
            "tdl",
            "drug_count",
            "publication_count",
            "ppi_count",
        ],
    ]
    .assign(tdl=lambda frame: frame["tdl"].astype(tdl_order))
    .sort_values(
        ["tdl", "drug_count", "gene_symbol"],
        ascending=[True, False, True],
    )
)
target_summary
